# explore_selection_for_a2

Notebook focussed on automating the process for selecting a2 data.

@author: David Clemens-Sewall

In [6]:
# imports
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from datetime import date

%matplotlib inline

In [2]:
# Read in a1 data
data_path = os.path.join('..', 'data', 'whoi_l_arm_transmittance')

project_names = ['20250730-r2-coring',
                 '20250730-r2-ssl',
                 ]

ls_df_a1 = []
for project_name in project_names:
    file_path = os.path.join(data_path,
                            project_name,
                            'contrasts_l_arm_' + project_name + '.a1.h5')
    ls_df_a1.append(pd.read_hdf(file_path))

df_a1 = pd.concat(ls_df_a1)

In [28]:
# Scratchwork for figuring out the pareto front
project_name = project_names[0]

proj_date = date(int(project_name[:4]), int(project_name[4:6]), int(project_name[6:8]))
proj_stat = project_name[10]
proj_loca = project_name.split('-')[-1]

# Extract the Inclination and Pressure values 
qry_str = ('timestamp_utc.dt.date==@proj_date & station==@proj_stat & location==@proj_loca'
           + ' & variable in ["InclV", "Pressure"]')
df_temp_T = df_a1.query(qry_str + ' & type=="T"')

# Go azimuth by azimuth and identify pareto front:
azs = df_temp_T.index.get_level_values('azimuth').unique()
pareto_timestamps = []
az = azs[0]
df_temp_T_az = df_temp_T.query('azimuth==@az')
df_temp_T_az = df_temp_T_az.reset_index().pivot(index='timestamp_utc', columns='variable', values='value')
df_temp_T_az = df_temp_T_az.sort_values('InclV').reset_index()

p_front = df_temp_T_az.index.values

i = 0
while i < p_front.size:
    ind = p_front[i]
    curr_press = df_temp_T_az.at[ind, 'Pressure']

    s_press = df_temp_T_az.loc[p_front[i+1:]]['Pressure']
    p_front = np.concatenate([p_front[:i+1], s_press.index[s_press<=curr_press].values])

    i += 1
    #break

pareto_timestampes.append(df_temp_T_az.loc[p_front, 'timestamp_utc'].values)

In [29]:


df_temp_T_az

variable,timestamp_utc,InclV,Pressure
0,2025-07-30 10:56:10+00:00,3.173577,0.161226
1,2025-07-30 10:55:04+00:00,3.936472,0.158944
2,2025-07-30 10:56:04+00:00,3.936472,0.165005
3,2025-07-30 10:56:13+00:00,3.936472,0.163186
4,2025-07-30 10:55:12+00:00,4.401998,0.148902
5,2025-07-30 10:55:58+00:00,5.151504,0.163186
6,2025-07-30 10:55:33+00:00,5.924826,0.148988
7,2025-07-30 10:55:26+00:00,11.053694,0.149886
8,2025-07-30 10:55:20+00:00,18.295692,0.147196


In [42]:
p_front = df_temp_T_az.index.values

i = 0
while i < p_front.size:
    ind = p_front[i]
    curr_press = df_temp_T_az.at[ind, 'Pressure']

    s_press = df_temp_T_az.loc[p_front[i+1:]]['Pressure']
    p_front = np.concatenate([p_front[:i+1], s_press.index[s_press<=curr_press].values])

    i += 1
    #break

In [45]:
df_temp_T_az.loc[p_front, 'timestamp_utc'].values

array(['2025-07-30T10:56:10.000000000', '2025-07-30T10:55:04.000000000',
       '2025-07-30T10:55:12.000000000', '2025-07-30T10:55:20.000000000'],
      dtype='datetime64[ns]')